# 02 Train Emulator (PINN from scratch)

Addestramento PINN indipendente dalla MLP: usa solo il dataset condiviso.


In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / 'PINN', _cwd.parent, _cwd.parent / 'PINN', _cwd.parent.parent]
_PROJECT_ROOT = next((p for p in _candidates if (p / 'src' / 'solsys_emulator').exists()), None)
if _PROJECT_ROOT is None:
    raise RuntimeError('Impossibile trovare la project root con src/solsys_emulator')

_SRC = _PROJECT_ROOT / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

print('Project root:', _PROJECT_ROOT)
print('Python executable:', sys.executable)



In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch

from solsys_emulator.config import DEFAULT_CHECKPOINT_PATH, DEFAULT_DATASET_PATH
from solsys_emulator.de440_dataset import load_dataset
from solsys_emulator.model import ModelConfig
from solsys_emulator.train import TrainConfig, train_emulator

RUN_PHYSICS_STAGE = True

dataset = load_dataset(DEFAULT_DATASET_PATH)
print('Dataset source:', dataset.get('metadata', {}).get('sample_source'))
print('States shape:', dataset['states'].shape)
print('Num samples:', len(dataset['times_seconds']))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pinn_ckpt = Path(DEFAULT_CHECKPOINT_PATH)
coarse_ckpt = pinn_ckpt.with_name('emulator_pinn_coarse.pt')
refine_ckpt = pinn_ckpt.with_name('emulator_pinn_refine.pt')
physics_ckpt = pinn_ckpt.with_name('emulator_pinn_physics.pt')

model_cfg = ModelConfig(
    num_bodies=len(dataset['bodies']),
    state_mode='position_only',
    hidden_dim=384,
    num_layers=6,
    fourier_features=40,
    min_frequency=0.25,
    max_frequency=48.0,
    frequency_spacing='log',
    head_layers=2,
    head_hidden_dim=160,
    dropout=0.0,
)

# Stage 1: coarse orbit fit on positions only.
# This skips autograd velocities and is much cheaper on CPU.
coarse_cfg = TrainConfig(
    epochs=450,
    batch_size=512,
    lr=3e-4,
    weight_decay=1e-6,
    val_fraction=0.10,
    split_mode='random',
    shuffle=True,
    early_stopping_patience=100,
    lr_scheduler='cosine',
    min_lr=1e-6,
    nbody_loss_weight=0.0,
    physics_loss_weight=0.0,
    smoothness_loss_weight=0.0,
    position_loss_weight=1.0,
    velocity_loss_weight=0.0,
    grad_clip_norm=1.0,
    compute_val_velocity_rmse=False,
    selection_metric='val_pos_rmse_km',
    force_chronological_for_derivatives=False,
    sort_train_for_derivatives=False,
    show_progress=True,
    device=device,
)

coarse = train_emulator(
    dataset,
    train_config=coarse_cfg,
    model_config=model_cfg,
    checkpoint_path=coarse_ckpt,
)
coarse_best_epoch = int(np.argmin(coarse['history']['val_pos_rmse_km'])) + 1
coarse_best_rmse = float(np.min(coarse['history']['val_pos_rmse_km']))

print('Coarse checkpoint:', coarse_ckpt)
print('Coarse best epoch:', coarse_best_epoch)
print('Coarse best val position RMSE [km]:', f'{coarse_best_rmse:,.2f}')

# Stage 2: refine positions + velocities via autograd, but on far fewer epochs.
refine_cfg = TrainConfig(
    epochs=220,
    batch_size=384,
    lr=3e-5,
    weight_decay=1e-6,
    val_fraction=0.10,
    split_mode='random',
    shuffle=True,
    early_stopping_patience=60,
    lr_scheduler='cosine',
    min_lr=1e-6,
    nbody_loss_weight=0.0,
    physics_loss_weight=0.0,
    smoothness_loss_weight=0.0,
    position_loss_weight=1.0,
    velocity_loss_weight=0.4,
    grad_clip_norm=1.0,
    compute_val_velocity_rmse=True,
    selection_metric='val_pos_rmse_km',
    force_chronological_for_derivatives=False,
    sort_train_for_derivatives=False,
    show_progress=True,
    device=device,
)

refine = train_emulator(
    dataset,
    train_config=refine_cfg,
    model_config=model_cfg,
    checkpoint_path=refine_ckpt,
    initial_checkpoint_path=coarse_ckpt,
)
refine_best_epoch = int(np.argmin(refine['history']['val_pos_rmse_km'])) + 1
refine_best_rmse = float(np.min(refine['history']['val_pos_rmse_km']))

print('Refine checkpoint:', refine_ckpt)
print('Refine best epoch:', refine_best_epoch)
print('Refine best val position RMSE [km]:', f'{refine_best_rmse:,.2f}')

physics = None
physics_best_epoch = None
physics_best_rmse = None
if RUN_PHYSICS_STAGE:
    # Stage 3: short physics fine-tuning on autograd acceleration residual.
    physics_cfg = TrainConfig(
        epochs=60,
        batch_size=384,
        lr=2e-6,
        weight_decay=1e-6,
        val_fraction=0.10,
        split_mode='random',
        shuffle=True,
        early_stopping_patience=12,
        lr_scheduler='cosine',
        min_lr=5e-7,
        nbody_loss_weight=1e-6,
        adaptive_nbody_balance=True,
        nbody_target_fraction=2e-3,
        nbody_balance_beta=0.9,
        nbody_balance_max_scale=1e6,
        nbody_batch_size=48,
        nbody_start_epoch=6,
        nbody_warmup_epochs=36,
        nbody_softening_km=80_000.0,
        nbody_relative_floor_km_s2=5e-4,
        physics_loss_weight=0.0,
        smoothness_loss_weight=0.0,
        position_loss_weight=1.0,
        velocity_loss_weight=0.0,
        grad_clip_norm=1.0,
        compute_val_velocity_rmse=False,
        selection_metric='val_pos_rmse_km',
        force_chronological_for_derivatives=False,
        sort_train_for_derivatives=False,
        show_progress=True,
        device=device,
    )
    physics = train_emulator(
        dataset,
        train_config=physics_cfg,
        model_config=model_cfg,
        checkpoint_path=physics_ckpt,
        initial_checkpoint_path=refine_ckpt,
    )
    physics_best_epoch = int(np.argmin(physics['history']['val_pos_rmse_km'])) + 1
    physics_best_rmse = float(np.min(physics['history']['val_pos_rmse_km']))

    print('Physics checkpoint:', physics_ckpt)
    print('Physics best epoch:', physics_best_epoch)
    print('Physics best val position RMSE [km]:', f'{physics_best_rmse:,.2f}')

# Select final checkpoint among stages that supervise full 6D states.
selected_stage = 'refine'
selected_ckpt = refine_ckpt
selected_rmse = refine_best_rmse
if physics is not None and physics_best_rmse is not None and physics_best_rmse < selected_rmse:
    selected_stage = 'physics'
    selected_ckpt = physics_ckpt
    selected_rmse = physics_best_rmse

shutil.copy2(selected_ckpt, pinn_ckpt)
print('Final PINN checkpoint:', pinn_ckpt)
print('Selected stage:', selected_stage)
print('Best val position RMSE [km]:', f'{selected_rmse:,.2f}')

plt.figure(figsize=(10, 4))
plt.plot(coarse['history']['train_loss'], label='coarse train(data)')
plt.plot(coarse['history']['val_loss'], label='coarse val(data)')
offset = len(coarse['history']['train_loss'])
plt.plot(range(offset, offset + len(refine['history']['train_loss'])), refine['history']['train_loss'], label='refine train(data)')
plt.plot(range(offset, offset + len(refine['history']['val_loss'])), refine['history']['val_loss'], label='refine val(data)')
if physics is not None:
    offset2 = offset + len(refine['history']['train_loss'])
    plt.plot(range(offset2, offset2 + len(physics['history']['train_loss'])), physics['history']['train_loss'], label='physics train(data)')
    plt.plot(range(offset2, offset2 + len(physics['history']['val_loss'])), physics['history']['val_loss'], label='physics val(data)')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('PINN training history')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(coarse['history']['val_pos_rmse_km'], label='coarse val position RMSE [km]')
plt.plot(range(len(coarse['history']['val_pos_rmse_km']), len(coarse['history']['val_pos_rmse_km']) + len(refine['history']['val_pos_rmse_km'])), refine['history']['val_pos_rmse_km'], label='refine val position RMSE [km]')
if physics is not None:
    offset2 = len(coarse['history']['val_pos_rmse_km']) + len(refine['history']['val_pos_rmse_km'])
    plt.plot(range(offset2, offset2 + len(physics['history']['val_pos_rmse_km'])), physics['history']['val_pos_rmse_km'], label='physics val position RMSE [km]')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('RMSE')
plt.title('PINN validation position RMSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

if physics is not None:
    plt.figure(figsize=(10, 3))
    nbody_raw = np.array(physics['history']['nbody_loss'])
    nbody_w = np.array(physics['history']['nbody_weight'])
    plt.plot(nbody_raw, label='physics nbody raw')
    plt.plot(np.maximum(1e-16, nbody_raw * nbody_w), label='physics nbody weighted')
    plt.yscale('log')
    plt.xlabel('epoch')
    plt.ylabel('loss contribution scale')
    plt.title('Physics-stage contribution diagnostics')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


